In [1]:
import os
import math
import numpy as np
from PIL import Image, ImageSequence, ImageDraw

# -----######-----######  MAIN FUNCTION  -----######-----######


def _gif_0912_circle1_GET_circ_gif(
    gif_paths,
    target_diameter_px=None,
    shrink_factor=0.98,
    out_suffix="_circ",
    verbose=True
):
    """
    Detects a circular logo/shape in a GIF, masks everything outside the circle
    (transparent), optionally rescales (interpolation), and exports a new GIF.

    Parameters
    ----------
    gif_paths : str or list
        Path or list of paths to input GIFs.
    target_diameter_px : int or None
        If None -> keep original circle size.
        If int -> rescale so that the circle diameter ≈ this value.
    shrink_factor : float
        Factor to slightly shrink the detected radius to avoid black ring edges.
    out_suffix : str
        Suffix to append to filename before extension.
    verbose : bool
        If True, prints TQM progress info.

    Returns
    -------
    results : list of dict
        One dict per GIF with:
            'src', 'out', 'circle_center', 'radius', 'scale', 'n_frames'
    """

    # Normalize input to list
    if isinstance(gif_paths, str):
        gif_paths = [gif_paths]

    results = []

    for gif_idx, gif_path in enumerate(gif_paths, start=1):
        if verbose:
            print("\n===================================================")
            print("Processing GIF [{}/{}]:".format(gif_idx, len(gif_paths)))
            print("  -> {}".format(gif_path))
            print("===================================================\n")

        if not os.path.isfile(gif_path):
            if verbose:
                print("  [WARN] File not found, skipping:", gif_path)
            continue

        im = Image.open(gif_path)
        w, h = im.size

        # -------------------------------------------------------
        # 1) Detect circle on first frame (non-black region)
        # -------------------------------------------------------
        first_frame = next(ImageSequence.Iterator(im)).convert("L")
        arr = np.array(first_frame, dtype=np.uint8)

        # Threshold: anything brighter than this is "non-black"
        # You can tweak this if your background is not pure black.
        thr = 10
        non_black = arr > thr

        if np.any(non_black):
            ys, xs = np.where(non_black)
            x_min, x_max = xs.min(), xs.max()
            y_min, y_max = ys.min(), ys.max()

            cx = 0.5 * (x_min + x_max)
            cy = 0.5 * (y_min + y_max)
            radius = 0.5 * min((x_max - x_min), (y_max - y_min))
        else:
            # Fallback: whole image circle
            if verbose:
                print("  [WARN] Non-black region not found; using full image circle.")
            cx = w / 2.0
            cy = h / 2.0
            radius = min(w, h) / 2.0

        # Slightly shrink radius so we don't catch black border artifacts
        radius = radius * shrink_factor

        if verbose:
            print("  Detected circle:")
            print("    center (cx, cy) ≈ ({:.1f}, {:.1f})".format(cx, cy))
            print("    radius ≈ {:.1f}px".format(radius))

        # -------------------------------------------------------
        # 2) Determine scale factor from target_diameter_px
        # -------------------------------------------------------
        circle_diameter = 2.0 * radius
        if target_diameter_px is None or target_diameter_px <= 0:
            scale = 1.0
            if verbose:
                print("  Keeping original resolution (no rescale).")
        else:
            scale = float(target_diameter_px) / float(circle_diameter)
            if verbose:
                print("  Target circle diameter: {}px".format(target_diameter_px))
                print("  Computed scale factor : {:.3f}x".format(scale))

        # -------------------------------------------------------
        # 3) Build frames list and prepare TQM bar
        # -------------------------------------------------------
        frames = [frame.copy() for frame in ImageSequence.Iterator(im)]
        n_frames = len(frames)

        if verbose:
            print("\n  Total frames:", n_frames)
            print("  Applying circular mask + upscale if needed...\n")

        frames_out = []
        durations = []

        for idx, frame in enumerate(frames, start=1):
            # Convert to RGBA so we can have per-pixel alpha
            rgba = frame.convert("RGBA")

            # Create circular mask
            mask = Image.new("L", (w, h), 0)
            draw = ImageDraw.Draw(mask)
            # Bounding box of ellipse
            bbox = (
                int(cx - radius),
                int(cy - radius),
                int(cx + radius),
                int(cy + radius),
            )
            draw.ellipse(bbox, fill=255)

            # Apply mask as alpha
            rgba.putalpha(mask)

            # Rescale if needed
            if scale != 1.0:
                new_w = max(1, int(round(w * scale)))
                new_h = max(1, int(round(h * scale)))
                rgba = rgba.resize((new_w, new_h), resample=Image.LANCZOS)

            frames_out.append(rgba)

            # Keep per-frame duration if available
            duration = frame.info.get("duration", im.info.get("duration", 40))
            durations.append(duration)

            # TQM progress bar
            if verbose:
                _tqm_0912_bar1_GET_print(idx, n_frames, prefix="  Frames processed")

        if verbose:
            print("\n\n  Masking + interpolation done.\n")

        # -------------------------------------------------------
        # 4) Save output GIF
        # -------------------------------------------------------
        in_dir = os.path.dirname(gif_path)
        in_base = os.path.splitext(os.path.basename(gif_path))[0]
        out_name = in_base + out_suffix + ".gif"
        out_path = os.path.join(in_dir, out_name)

        # GIF constraints: Pillow will handle palette + transparency conversion.
        # We set disposal=2 for proper redraw of each frame.
        save_kwargs = {
            "save_all": True,
            "append_images": frames_out[1:],
            "loop": 0,
            "duration": durations,
            "disposal": 2,
            "optimize": False,
        }

        # Convert first frame to P-mode GIF, others will be auto-quantized
        first_out = frames_out[0].convert("RGBA")
        first_out.save(out_path, **save_kwargs)

        if verbose:
            print("  Output saved to:")
            print("    {}".format(out_path))

        results.append(
            {
                "src": gif_path,
                "out": out_path,
                "circle_center": (cx, cy),
                "radius": radius,
                "scale": scale,
                "n_frames": n_frames,
            }
        )

        im.close()

    if verbose:
        print("\nAll GIFs processed.")
    return results


# -----------------------------------------------------------
#  TQM BAR (ASCII PROGRESS BAR)
# -----------------------------------------------------------

def _tqm_0912_bar1_GET_print(current, total, prefix="", bar_len=30):
    """
    Simple ASCII TQM-style progress bar.
    """
    if total <= 0:
        total = 1
    frac = float(current) / float(total)
    frac = max(0.0, min(1.0, frac))

    filled_len = int(round(bar_len * frac))
    bar = "#" * filled_len + "-" * (bar_len - filled_len)
    msg = "\r{} |{}| {:>3d}% ({}/{})".format(
        prefix, bar, int(frac * 100), current, total
    )
    print(msg, end="", flush=True)


In [ ]:
if __name__ == "__main__":
    print("\n==============================================")
    print("  CIRCULAR GIF MASK + QUALITY INTERPOLATION")
    print("==============================================\n")

    in_path = input("Enter path to input GIF:\n> ").strip().strip('"').strip("'")
    if not in_path:
        raise ValueError("No input GIF path provided.")

    print("\nChoose quality / size for the circular GIF:")
    print("  1 = Keep original size (no upscale)")
    print("  2 = Upscale circle to 512px diameter")
    print("  3 = Upscale circle to 1024px diameter")
    print("  4 = Custom diameter (you type the px)")
    choice = input("\nQuality option [1/2/3/4]: ").strip()

    target_diam = None

    if choice == "1" or choice == "":
        target_diam = None
    elif choice == "2":
        target_diam = 512
    elif choice == "3":
        target_diam = 1024
    elif choice == "4":
        custom = input("Enter target circle diameter in pixels (e.g. 700): ").strip()
        if custom:
            target_diam = int(custom)
        else:
            target_diam = None
    else:
        print("Unrecognized choice, keeping original size.")
        target_diam = None

    print("\nRunning circular detection + masking + interpolation...\n")
    res = _gif_0912_circle1_GET_circ_gif(
        gif_paths=in_path,
        target_diameter_px=target_diam,
        shrink_factor=0.98,   # you can tweak this if edges show black
        out_suffix="_circ",   # suffix for output
        verbose=True
    )

    print("\nRESULTS:")
    for r in res:
        print("  Source GIF :", r["src"])
        print("  Output GIF :", r["out"])
        print("  Center     :", r["circle_center"])
        print("  Radius (px): {:.2f}".format(r["radius"]))
        print("  Scale (x)  :", "{:.3f}".format(r["scale"]))
        print("  Frames     :", r["n_frames"])
        print("")



  CIRCULAR GIF MASK + QUALITY INTERPOLATION



In [1]:
#========================================
# 0_FNS
#========================================
import os
import subprocess
from pathlib import Path

#-----######-----######  CORE FN  -----######-----######
def _wav_2201_mq_GET_mp3(
    wav_path,
    out_dir="",
    bitrate="192k",
    sr=44100,
    channels=2,
    overwrite="y",
):
    """
    Fast WAV -> MP3 (mid quality) using ffmpeg.

    Inputs:
      wav_path: path to a .wav file
      out_dir: output folder ("" -> same folder as wav)
      bitrate: e.g. "160k", "192k" (mid quality)
      sr: sample rate (keeps output consistent)
      channels: 2=stereo, 1=mono
      overwrite: "y" or "n"

    Returns:
      mp3_path (str)
    """
    wav_path = Path(os.path.expanduser(str(wav_path))).resolve()
    if not wav_path.exists():
        raise FileNotFoundError(f"File not found: {wav_path}")
    if wav_path.suffix.lower() != ".wav":
        raise ValueError(f"Expected .wav file, got: {wav_path.suffix}")

    if out_dir:
        out_dir = Path(os.path.expanduser(str(out_dir))).resolve()
    else:
        out_dir = wav_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    mp3_path = (out_dir / f"{wav_path.stem}.mp3").resolve()

    ow_flag = "-y" if str(overwrite).lower().startswith("y") else "-n"

    cmd = [
        "ffmpeg",
        ow_flag,
        "-hide_banner",
        "-loglevel", "error",
        "-i", str(wav_path),
        "-vn",
        "-c:a", "libmp3lame",
        "-b:a", str(bitrate),
        "-ar", str(int(sr)),
        "-ac", str(int(channels)),
        str(mp3_path),
    ]

    try:
        subprocess.run(cmd, check=True)
    except FileNotFoundError:
        raise RuntimeError(
            "ffmpeg not found. Install it first:\n"
            "  brew install ffmpeg"
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"ffmpeg failed: {e}")

    return str(mp3_path)


In [3]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
mp3_path = _wav_2201_mq_GET_mp3(
    wav_path="/Users/yerik/Desktop/__LOG__/_25DJSet__11-07_Detroit_Techno/det-techno_yodj_rec.WAV",
    out_dir="",        # or "/path/to/output_folder"
    bitrate="192k",    # mid quality
    sr=44100,
    channels=2,        # set 1 for mono if you want smaller/faster
    overwrite="y",
)
print(mp3_path)


/Users/yerik/Desktop/__LOG__/_25DJSet__11-07_Detroit_Techno/det-techno_yodj_rec.mp3
